# Notebook Tuning BERTopic vs LDA (Web Form Aligned)

Notebook ini dibuat untuk eksplorasi parameter tuning yang **selaras dengan form pengaturan di website Laravel** dan tetap memakai komponen inti FastAPI agar hasil konsisten saat parameter sama.

Alur yang dipakai:
1. Preprocessing
2. BERTopic loop (embedding dihitung sekali, dipakai ulang semua skenario)
3. LDA loop
4. Komparasi model
5. Ambil best parameter + retrain best model
6. Visualisasi (wordcloud, distribusi topik, CV, TD)
7. DTA (emerging, stable, declining)

Semua tahap penting mengekspor artefak CSV/JSON ke folder hasil run.

In [ ]:
import itertools
import json
import math
import random
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    from wordcloud import WordCloud
except Exception:
    WordCloud = None

from IPython.display import display

from app.core import database
from app.ml.bertopic_trainer import BERTopicTrainer
from app.ml.evaluator import TopicEvaluator
from app.ml.lda_trainer import LDATrainer
from app.models.schemas import (
    BERTopicHyperparameters,
    HDBSCANHyperparameters,
    LDAHyperparameters,
    UMAPHyperparameters,
)
from app.services.preprocessing import TextPreprocessor

GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

CWD = Path.cwd()
if (CWD / "app").exists():
    FASTAPI_ROOT = CWD
elif (CWD.parent / "app").exists():
    FASTAPI_ROOT = CWD.parent
else:
    raise RuntimeError("Jalankan notebook dari folder fastapi atau subfolder di dalamnya.")

RUN_TS = datetime.now().strftime("%Y%m%d-%H%M%S")
ARTIFACT_ROOT = FASTAPI_ROOT / "data" / "results" / "notebook_tuning_web_form"
RUN_DIR = ARTIFACT_ROOT / f"run_{RUN_TS}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)

print("FASTAPI_ROOT:", FASTAPI_ROOT)
print("RUN_DIR:", RUN_DIR)
print("WordCloud available:", WordCloud is not None)

## 1) Konfigurasi Parameter (Selaras Form Website)

Isi `WEB_FORM_*_BASE` dengan nilai yang sama seperti di form Laravel jika ingin reproduksi 1:1.

`*_TUNING_SPACE` bisa diisi satu nilai (untuk reproduksi) atau beberapa nilai (untuk tuning loop).

In [ ]:
# Base params yang mengikuti field form Laravel + default FastAPI current.
WEB_FORM_BERTOPIC_BASE = {
    "embedding_model": "denaya/indoSBERT-large",
    "min_topic_size": 10,
    "nr_topics": 8,
    "top_n_words": 15,
    "n_gram_range": [1, 2],
    "vectorizer_min_df": 2,
    "vectorizer_max_df": 0.95,
    "vectorizer_token_pattern": r"(?u)\b\w{3,}\b",
    "vectorizer_fallback_min_df": 1,
    "vectorizer_fallback_max_df": 1.0,
    "coherence_type": "c_v",
    "coherence_tokenization": "vectorizer",
    "coherence_dict_no_below": 3,
    "coherence_dict_no_above": 0.95,
    "reduce_outliers": True,
    "reduce_outliers_threshold_ctfidf": 0.1,
    "reduce_outliers_use_distributions": True,
    "reduce_outliers_threshold_distributions": 0.05,
    "use_mmr_representation": True,
    "mmr_diversity": 0.3,
    "embedding_batch_size": 16,
    "seed": 42,
    "umap_params": {
        "n_neighbors": 40,
        "n_components": 5,
        "min_dist": 0.0,
        "metric": "cosine",
        "random_state": 42,
    },
    "hdbscan_params": {
        "min_cluster_size": 8,
        "min_samples": 1,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
    },
}

WEB_FORM_LDA_BASE = {
    "num_topics": 12,
    "passes": 20,
    "iterations": 300,
    "chunksize": 100,
    "random_state": 42,
    "alpha": "asymmetric",
    "eta": None,
    "no_below": 2,
    "no_above": 0.95,
}

# Isi list lebih dari satu nilai untuk tuning eksploratif.
BERTOPIC_TUNING_SPACE = {
    "min_topic_size": [WEB_FORM_BERTOPIC_BASE["min_topic_size"]],
    "nr_topics": [WEB_FORM_BERTOPIC_BASE["nr_topics"]],
    "top_n_words": [WEB_FORM_BERTOPIC_BASE["top_n_words"]],
    "n_gram_range": [tuple(WEB_FORM_BERTOPIC_BASE["n_gram_range"])],
    "umap_n_neighbors": [WEB_FORM_BERTOPIC_BASE["umap_params"]["n_neighbors"]],
    "umap_n_components": [WEB_FORM_BERTOPIC_BASE["umap_params"]["n_components"]],
    "umap_metric": [WEB_FORM_BERTOPIC_BASE["umap_params"]["metric"]],
    "hdbscan_min_cluster_size": [WEB_FORM_BERTOPIC_BASE["hdbscan_params"]["min_cluster_size"]],
    "hdbscan_min_samples": [WEB_FORM_BERTOPIC_BASE["hdbscan_params"]["min_samples"]],
    "hdbscan_metric": [WEB_FORM_BERTOPIC_BASE["hdbscan_params"]["metric"]],
}

LDA_TUNING_SPACE = {
    "num_topics": [WEB_FORM_LDA_BASE["num_topics"]],
    "passes": [WEB_FORM_LDA_BASE["passes"]],
    "iterations": [WEB_FORM_LDA_BASE["iterations"]],
    "chunksize": [WEB_FORM_LDA_BASE["chunksize"]],
    "random_state": [WEB_FORM_LDA_BASE["random_state"]],
    "alpha": [WEB_FORM_LDA_BASE["alpha"]],
    "eta": [WEB_FORM_LDA_BASE["eta"]],
    "no_below": [WEB_FORM_LDA_BASE["no_below"]],
    "no_above": [WEB_FORM_LDA_BASE["no_above"]],
}

MAX_BERTOPIC_SCENARIOS = 12
MAX_LDA_SCENARIOS = 12
SCENARIO_SEED = 42

print("BERTopic base params loaded")
print("LDA base params loaded")

## 2) Preprocessing + Export Before/After CSV

Bagian ini meniru aturan pipeline preprocessing FastAPI (drop null abstract, dedup abstract, filter year, dual output cleaned/processed), lalu ekspor CSV per tahap untuk pelaporan.

In [ ]:
preprocessor = TextPreprocessor(
    remove_stopwords=True,
    use_stemming=True,
    min_word_length=3,
    language="indonesian",
)

raw_df = database.load_abstracts_from_db()
raw_path = RUN_DIR / "preprocess_step0_raw_source.csv"
raw_df.to_csv(raw_path, index=False)

stage_df = raw_df.copy()
dropna_mask = stage_df["abstract"].isna() | (stage_df["abstract"].astype(str).str.strip() == "")
dropna_df = stage_df.loc[dropna_mask].copy()
stage_df = stage_df.loc[~dropna_mask].copy()

duplicate_mask = stage_df.duplicated(subset=["abstract"], keep="first")
duplicate_df = stage_df.loc[duplicate_mask].copy()
stage_df = stage_df.loc[~duplicate_mask].copy()

year_series = pd.to_numeric(stage_df["year"], errors="coerce")
year_mask = year_series.between(2018, 2026)
year_drop_df = stage_df.loc[~year_mask].copy()
stage_df = stage_df.loc[year_mask].copy()
stage_df["year"] = pd.to_numeric(stage_df["year"], errors="coerce").astype("Int64")
stage_df = stage_df.reset_index(drop=True)

filtered_path = RUN_DIR / "preprocess_step1_filtered.csv"
stage_df.to_csv(filtered_path, index=False)

ids_before = set(stage_df["id"].astype(int).tolist()) if "id" in stage_df.columns else set()

preprocessed_df = preprocessor.preprocess_dataframe(
    stage_df,
    text_column="abstract",
    title_column="title",
    conclusion_column="conclusion",
)

ids_after = set(preprocessed_df["id"].astype(int).tolist()) if "id" in preprocessed_df.columns else set()
empty_after_ids = ids_before - ids_after
empty_after_df = stage_df[stage_df["id"].isin(empty_after_ids)].copy() if empty_after_ids else stage_df.head(0).copy()

dual_path = RUN_DIR / "preprocess_step2_dual_preprocessed.csv"
preprocessed_df.to_csv(dual_path, index=False)

before_cols = [c for c in ["id", "title", "year", "abstract", "conclusion"] if c in stage_df.columns]
after_cols = [c for c in ["id", "combined_text", "cleaned_text", "processed_text"] if c in preprocessed_df.columns]

before_after_df = stage_df[before_cols].merge(preprocessed_df[after_cols], on="id", how="inner")
before_after_df["token_before"] = before_after_df["abstract"].fillna("").astype(str).str.split().apply(len)
before_after_df["token_cleaned"] = before_after_df["cleaned_text"].fillna("").astype(str).str.split().apply(len)
before_after_df["token_processed"] = before_after_df["processed_text"].fillna("").astype(str).str.split().apply(len)
before_after_df["token_reduction_processed"] = before_after_df["token_before"] - before_after_df["token_processed"]

before_after_path = RUN_DIR / "preprocess_step3_before_after_audit.csv"
before_after_df.to_csv(before_after_path, index=False)

drop_summary_df = pd.DataFrame([
    {"reason": "dropna_abstract", "count": int(len(dropna_df))},
    {"reason": "duplicate_abstract", "count": int(len(duplicate_df))},
    {"reason": "year_out_of_range", "count": int(len(year_drop_df))},
    {"reason": "empty_after_preprocessing", "count": int(len(empty_after_df))},
])
drop_summary_path = RUN_DIR / "preprocess_drop_summary.csv"
drop_summary_df.to_csv(drop_summary_path, index=False)

if preprocessed_df.empty:
    raise ValueError("Dataset kosong setelah preprocessing.")

bertopic_docs = preprocessed_df["cleaned_text"].astype(str).tolist()
lda_docs = preprocessed_df["processed_text"].astype(str).tolist()
document_ids = preprocessed_df["id"].astype(int).tolist()
timestamps = preprocessed_df["year"].astype(int).tolist() if preprocessed_df["year"].notna().all() else None

print("Raw rows:", len(raw_df))
print("Filtered rows:", len(stage_df))
print("Final preprocessed rows:", len(preprocessed_df))
display(drop_summary_df)
display(before_after_df.head(3))

## 3) Loop Tuning BERTopic dan LDA

- BERTopic: embedding dihitung sekali di awal, lalu dipakai ulang ke semua skenario.
- LDA: loop standar per skenario.
- Semua hasil loop diekspor ke CSV.

In [ ]:
def build_combinations(space_dict):
    keys = list(space_dict.keys())
    value_lists = []
    for key in keys:
        value = space_dict[key]
        if isinstance(value, list):
            value_lists.append(value)
        else:
            value_lists.append([value])
    return [dict(zip(keys, values)) for values in itertools.product(*value_lists)]


def sample_combinations(combos, max_items, seed):
    if max_items is None or len(combos) <= max_items:
        return combos
    rng = random.Random(seed)
    return rng.sample(combos, k=max_items)


def normalize_nr_topics(value):
    if value is None:
        return None
    text = str(value).strip().lower()
    if text in {"", "none", "null"}:
        return None
    if text == "auto":
        return "auto"
    return max(8, int(float(text)))


def make_bertopic_params(combo):
    ngram = combo["n_gram_range"]
    if not isinstance(ngram, (list, tuple)) or len(ngram) != 2:
        raise ValueError(f"Invalid n_gram_range: {ngram}")

    return BERTopicHyperparameters(
        embedding_model=WEB_FORM_BERTOPIC_BASE["embedding_model"],
        min_topic_size=int(combo["min_topic_size"]),
        nr_topics=normalize_nr_topics(combo["nr_topics"]),
        top_n_words=int(combo["top_n_words"]),
        n_gram_range=[int(ngram[0]), int(ngram[1])],
        vectorizer_min_df=WEB_FORM_BERTOPIC_BASE["vectorizer_min_df"],
        vectorizer_max_df=WEB_FORM_BERTOPIC_BASE["vectorizer_max_df"],
        vectorizer_token_pattern=WEB_FORM_BERTOPIC_BASE["vectorizer_token_pattern"],
        vectorizer_fallback_min_df=WEB_FORM_BERTOPIC_BASE["vectorizer_fallback_min_df"],
        vectorizer_fallback_max_df=WEB_FORM_BERTOPIC_BASE["vectorizer_fallback_max_df"],
        coherence_type=WEB_FORM_BERTOPIC_BASE["coherence_type"],
        coherence_tokenization=WEB_FORM_BERTOPIC_BASE["coherence_tokenization"],
        coherence_dict_no_below=WEB_FORM_BERTOPIC_BASE["coherence_dict_no_below"],
        coherence_dict_no_above=WEB_FORM_BERTOPIC_BASE["coherence_dict_no_above"],
        reduce_outliers=WEB_FORM_BERTOPIC_BASE["reduce_outliers"],
        reduce_outliers_threshold_ctfidf=WEB_FORM_BERTOPIC_BASE["reduce_outliers_threshold_ctfidf"],
        reduce_outliers_use_distributions=WEB_FORM_BERTOPIC_BASE["reduce_outliers_use_distributions"],
        reduce_outliers_threshold_distributions=WEB_FORM_BERTOPIC_BASE["reduce_outliers_threshold_distributions"],
        use_mmr_representation=WEB_FORM_BERTOPIC_BASE["use_mmr_representation"],
        mmr_diversity=WEB_FORM_BERTOPIC_BASE["mmr_diversity"],
        embedding_batch_size=WEB_FORM_BERTOPIC_BASE["embedding_batch_size"],
        seed=WEB_FORM_BERTOPIC_BASE["seed"],
        umap_params=UMAPHyperparameters(
            n_neighbors=int(combo["umap_n_neighbors"]),
            n_components=int(combo["umap_n_components"]),
            min_dist=float(WEB_FORM_BERTOPIC_BASE["umap_params"]["min_dist"]),
            metric=str(combo["umap_metric"]),
            random_state=int(WEB_FORM_BERTOPIC_BASE["umap_params"]["random_state"]),
        ),
        hdbscan_params=HDBSCANHyperparameters(
            min_cluster_size=int(combo["hdbscan_min_cluster_size"]),
            min_samples=None if combo["hdbscan_min_samples"] is None else int(combo["hdbscan_min_samples"]),
            metric=str(combo["hdbscan_metric"]),
            cluster_selection_method=WEB_FORM_BERTOPIC_BASE["hdbscan_params"]["cluster_selection_method"],
        ),
    )


def make_lda_params(combo):
    return LDAHyperparameters(
        num_topics=int(combo["num_topics"]),
        passes=int(combo["passes"]),
        iterations=int(combo["iterations"]),
        chunksize=int(combo["chunksize"]),
        random_state=int(combo["random_state"]),
        alpha=combo["alpha"],
        eta=combo["eta"],
        no_below=int(combo["no_below"]),
        no_above=float(combo["no_above"]),
    )


all_bertopic_combos = build_combinations(BERTOPIC_TUNING_SPACE)
all_lda_combos = build_combinations(LDA_TUNING_SPACE)

sampled_bertopic_combos = sample_combinations(all_bertopic_combos, MAX_BERTOPIC_SCENARIOS, SCENARIO_SEED)
sampled_lda_combos = sample_combinations(all_lda_combos, MAX_LDA_SCENARIOS, SCENARIO_SEED)

bertopic_scenarios = [
    {
        "scenario_id": f"bertopic_{idx:03d}",
        "combo": combo,
        "params": make_bertopic_params(combo),
    }
    for idx, combo in enumerate(sampled_bertopic_combos, start=1)
]

lda_scenarios = [
    {
        "scenario_id": f"lda_{idx:03d}",
        "combo": combo,
        "params": make_lda_params(combo),
    }
    for idx, combo in enumerate(sampled_lda_combos, start=1)
]

scenario_df = pd.DataFrame(
    [{"model": "BERTopic", "scenario_id": s["scenario_id"], "combo": json.dumps(s["combo"], ensure_ascii=False)} for s in bertopic_scenarios]
    + [{"model": "LDA", "scenario_id": s["scenario_id"], "combo": json.dumps(s["combo"], ensure_ascii=False)} for s in lda_scenarios]
)
scenario_path = RUN_DIR / "scenario_space.csv"
scenario_df.to_csv(scenario_path, index=False)

print("BERTopic scenarios:", len(bertopic_scenarios))
print("LDA scenarios:", len(lda_scenarios))
display(scenario_df.head(10))

evaluator = TopicEvaluator()

# BERTopic: compute embeddings ONCE, reuse across all scenarios.
shared_embeddings = None
if bertopic_scenarios:
    embedding_models = {s["params"].embedding_model for s in bertopic_scenarios}
    if len(embedding_models) != 1:
        raise ValueError("Untuk shared embedding, semua skenario BERTopic harus memakai embedding_model yang sama.")

    print("Computing BERTopic embeddings once...")
    embedding_trainer = BERTopicTrainer(params=bertopic_scenarios[0]["params"])
    shared_embeddings = embedding_trainer.compute_embeddings(bertopic_docs)
    np.save(RUN_DIR / "bertopic_shared_embeddings.npy", shared_embeddings)
    print("Shared embeddings shape:", shared_embeddings.shape)

bertopic_rows = []
bertopic_success_payloads = []

for idx, scenario in enumerate(bertopic_scenarios, start=1):
    print(f"[BERTopic {idx}/{len(bertopic_scenarios)}] {scenario['scenario_id']}")
    row = {
        "model": "BERTopic",
        "scenario_id": scenario["scenario_id"],
        "status": "ok",
    }

    try:
        trainer = BERTopicTrainer(params=scenario["params"])
        train_result = trainer.train(
            documents=bertopic_docs,
            embeddings=shared_embeddings,
            timestamps=timestamps,
            document_ids=document_ids,
        )

        metrics = evaluator.evaluate_bertopic(
            model=trainer.model,
            documents=bertopic_docs,
            topics=trainer.topics,
            vectorizer_model=trainer.vectorizer_model,
            coherence_type=trainer.params.coherence_type,
            coherence_tokenization=trainer.params.coherence_tokenization,
            coherence_dict_no_below=trainer.params.coherence_dict_no_below,
            coherence_dict_no_above=trainer.params.coherence_dict_no_above,
            top_n_words=trainer.params.top_n_words,
        )

        objective_score = float(metrics.get("score_cv_td", metrics.get("score", 0.0)) or 0.0)

        row.update({
            "objective_score": objective_score,
            "coherence_cv": metrics.get("coherence_cv"),
            "topic_diversity": metrics.get("topic_diversity"),
            "num_topics": metrics.get("num_topics", train_result.get("num_topics")),
            "num_outliers": metrics.get("num_outliers", train_result.get("num_outliers")),
            "outlier_pct": metrics.get("outlier_pct"),
            "runtime_seconds": train_result.get("training_duration_seconds"),
            "params_json": json.dumps(scenario["params"].model_dump(), ensure_ascii=False),
        })

        bertopic_success_payloads.append({
            "scenario": scenario,
            "trainer": trainer,
            "train_result": train_result,
            "metrics": metrics,
            "objective_score": objective_score,
        })

    except Exception as exc:
        row["status"] = "failed"
        row["error"] = str(exc)
        row["params_json"] = json.dumps(scenario["params"].model_dump(), ensure_ascii=False)

    bertopic_rows.append(row)

bertopic_results_df = pd.DataFrame(bertopic_rows)
if not bertopic_results_df.empty and "objective_score" in bertopic_results_df.columns:
    bertopic_results_df["objective_score"] = pd.to_numeric(bertopic_results_df["objective_score"], errors="coerce")
    bertopic_results_df = bertopic_results_df.sort_values("objective_score", ascending=False, na_position="last").reset_index(drop=True)

bertopic_results_path = RUN_DIR / "bertopic_loop_results.csv"
bertopic_results_df.to_csv(bertopic_results_path, index=False)

best_bertopic_payload = None
if bertopic_success_payloads:
    best_bertopic_payload = max(bertopic_success_payloads, key=lambda x: float(x.get("objective_score", 0.0) or 0.0))


lda_rows = []
lda_success_payloads = []

for idx, scenario in enumerate(lda_scenarios, start=1):
    print(f"[LDA {idx}/{len(lda_scenarios)}] {scenario['scenario_id']}")
    row = {
        "model": "LDA",
        "scenario_id": scenario["scenario_id"],
        "status": "ok",
    }

    try:
        trainer = LDATrainer(params=scenario["params"])
        train_result = trainer.train(documents=lda_docs, timestamps=timestamps)

        metrics = evaluator.evaluate_lda(
            model=trainer.model,
            tokenized_docs=trainer.tokenized_docs,
            dictionary=trainer.dictionary,
        )

        objective_score = float(metrics.get("score", 0.0) or 0.0)

        row.update({
            "objective_score": objective_score,
            "coherence_cv": metrics.get("coherence_cv"),
            "topic_diversity": metrics.get("topic_diversity"),
            "num_topics": metrics.get("num_topics", train_result.get("num_topics")),
            "runtime_seconds": train_result.get("training_duration_seconds"),
            "params_json": json.dumps(scenario["params"].model_dump(), ensure_ascii=False),
        })

        lda_success_payloads.append({
            "scenario": scenario,
            "trainer": trainer,
            "train_result": train_result,
            "metrics": metrics,
            "objective_score": objective_score,
        })

    except Exception as exc:
        row["status"] = "failed"
        row["error"] = str(exc)
        row["params_json"] = json.dumps(scenario["params"].model_dump(), ensure_ascii=False)

    lda_rows.append(row)

lda_results_df = pd.DataFrame(lda_rows)
if not lda_results_df.empty and "objective_score" in lda_results_df.columns:
    lda_results_df["objective_score"] = pd.to_numeric(lda_results_df["objective_score"], errors="coerce")
    lda_results_df = lda_results_df.sort_values("objective_score", ascending=False, na_position="last").reset_index(drop=True)

lda_results_path = RUN_DIR / "lda_loop_results.csv"
lda_results_df.to_csv(lda_results_path, index=False)

best_lda_payload = None
if lda_success_payloads:
    best_lda_payload = max(lda_success_payloads, key=lambda x: float(x.get("objective_score", 0.0) or 0.0))

comparison_rows = []
if not bertopic_results_df.empty:
    ok_b = bertopic_results_df[bertopic_results_df["status"] == "ok"].copy()
    for _, r in ok_b.iterrows():
        comparison_rows.append({
            "model": "BERTopic",
            "scenario_id": r.get("scenario_id"),
            "objective_score": r.get("objective_score"),
            "coherence_cv": r.get("coherence_cv"),
            "topic_diversity": r.get("topic_diversity"),
            "runtime_seconds": r.get("runtime_seconds"),
            "num_topics": r.get("num_topics"),
        })

if not lda_results_df.empty:
    ok_l = lda_results_df[lda_results_df["status"] == "ok"].copy()
    for _, r in ok_l.iterrows():
        comparison_rows.append({
            "model": "LDA",
            "scenario_id": r.get("scenario_id"),
            "objective_score": r.get("objective_score"),
            "coherence_cv": r.get("coherence_cv"),
            "topic_diversity": r.get("topic_diversity"),
            "runtime_seconds": r.get("runtime_seconds"),
            "num_topics": r.get("num_topics"),
        })

leaderboard_df = pd.DataFrame(comparison_rows)

def _minmax(series):
    values = pd.to_numeric(series, errors="coerce")
    cmin = values.min()
    cmax = values.max()
    if pd.isna(cmin) or pd.isna(cmax) or cmin == cmax:
        return pd.Series(0.5, index=values.index, dtype=float)
    return (values - cmin) / (cmax - cmin)

if not leaderboard_df.empty:
    leaderboard_df["objective_norm"] = _minmax(leaderboard_df["objective_score"])
    leaderboard_df["coherence_norm"] = _minmax(leaderboard_df["coherence_cv"])
    leaderboard_df["diversity_norm"] = _minmax(leaderboard_df["topic_diversity"])
    leaderboard_df["runtime_norm"] = 1 - _minmax(leaderboard_df["runtime_seconds"])

    leaderboard_df["composite_score"] = (
        0.45 * leaderboard_df["objective_norm"]
        + 0.30 * leaderboard_df["coherence_norm"]
        + 0.20 * leaderboard_df["diversity_norm"]
        + 0.05 * leaderboard_df["runtime_norm"]
    )

    leaderboard_df = leaderboard_df.sort_values(
        ["composite_score", "objective_score", "coherence_cv"],
        ascending=[False, False, False],
        na_position="last",
    ).reset_index(drop=True)
    leaderboard_df["rank"] = np.arange(1, len(leaderboard_df) + 1)

comparison_path = RUN_DIR / "model_comparison_leaderboard.csv"
leaderboard_df.to_csv(comparison_path, index=False)

print("BERTopic loop rows:", len(bertopic_results_df))
print("LDA loop rows:", len(lda_results_df))
display(leaderboard_df.head(10))

## 4) Retrain Best Model + Simpan Artefak

Notebook ini retrain best BERTopic dan best LDA dari hasil loop, lalu menyimpan model + best config + keyword topic ke CSV/JSON.

In [ ]:
def to_serializable(obj):
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_serializable(payload), f, ensure_ascii=False, indent=2)


def save_bertopic_with_fallback(trainer, job_id):
    try:
        return trainer.save_model(job_id)
    except PermissionError as exc:
        fallback_dir = RUN_DIR / "saved_models" / f"bertopic_{job_id}"
        fallback_dir.mkdir(parents=True, exist_ok=True)
        trainer.model.save(str(fallback_dir / "model"), serialization="safetensors", save_ctfidf=True)
        if getattr(trainer, "embeddings", None) is not None:
            np.save(str(fallback_dir / "embeddings.npy"), trainer.embeddings)
        write_json(
            fallback_dir / "metadata.json",
            {
                "job_id": job_id,
                "model_type": "bertopic",
                "save_mode": "fallback_notebook",
                "reason": str(exc),
                "saved_at": datetime.now().isoformat(),
            },
        )
        return str(fallback_dir)


def save_lda_with_fallback(trainer, job_id):
    try:
        return trainer.save_model(job_id)
    except PermissionError as exc:
        fallback_dir = RUN_DIR / "saved_models" / f"lda_{job_id}"
        fallback_dir.mkdir(parents=True, exist_ok=True)
        trainer.model.save(str(fallback_dir / "lda_model"))
        trainer.dictionary.save(str(fallback_dir / "dictionary.dict"))
        write_json(
            fallback_dir / "metadata.json",
            {
                "job_id": job_id,
                "model_type": "lda",
                "save_mode": "fallback_notebook",
                "reason": str(exc),
                "saved_at": datetime.now().isoformat(),
            },
        )
        return str(fallback_dir)


final_bertopic = None
final_lda = None

if best_bertopic_payload is not None:
    params = best_bertopic_payload["scenario"]["params"]
    trainer = BERTopicTrainer(params=params)
    train_result = trainer.train(
        documents=bertopic_docs,
        embeddings=shared_embeddings,
        timestamps=timestamps,
        document_ids=document_ids,
    )
    metrics = evaluator.evaluate_bertopic(
        model=trainer.model,
        documents=bertopic_docs,
        topics=trainer.topics,
        vectorizer_model=trainer.vectorizer_model,
        coherence_type=trainer.params.coherence_type,
        coherence_tokenization=trainer.params.coherence_tokenization,
        coherence_dict_no_below=trainer.params.coherence_dict_no_below,
        coherence_dict_no_above=trainer.params.coherence_dict_no_above,
        top_n_words=trainer.params.top_n_words,
    )
    model_path = save_bertopic_with_fallback(trainer, f"nb_best_bertopic_{RUN_TS}")
    final_bertopic = {
        "trainer": trainer,
        "params": params,
        "train_result": train_result,
        "metrics": metrics,
        "model_path": model_path,
        "scenario_id": best_bertopic_payload["scenario"]["scenario_id"],
    }

if best_lda_payload is not None:
    params = best_lda_payload["scenario"]["params"]
    trainer = LDATrainer(params=params)
    train_result = trainer.train(documents=lda_docs, timestamps=timestamps)
    metrics = evaluator.evaluate_lda(
        model=trainer.model,
        tokenized_docs=trainer.tokenized_docs,
        dictionary=trainer.dictionary,
    )
    model_path = save_lda_with_fallback(trainer, f"nb_best_lda_{RUN_TS}")
    final_lda = {
        "trainer": trainer,
        "params": params,
        "train_result": train_result,
        "metrics": metrics,
        "model_path": model_path,
        "scenario_id": best_lda_payload["scenario"]["scenario_id"],
    }

keyword_frames = []
if final_bertopic is not None:
    b_rows = []
    b_info = final_bertopic["trainer"].model.get_topic_info().copy()
    b_count_map = {}
    if "Topic" in b_info.columns and "Count" in b_info.columns:
        for _, rr in b_info.iterrows():
            topic_id = int(rr.get("Topic", -1))
            if topic_id != -1:
                b_count_map[topic_id] = int(rr.get("Count", 0))

    for topic_id, words_scores in (final_bertopic["trainer"].model.get_topics() or {}).items():
        topic_id = int(topic_id)
        if topic_id == -1:
            continue
        for rank, (word, score) in enumerate((words_scores or [])[:15], start=1):
            b_rows.append({
                "model": "BERTopic",
                "topic_id": topic_id,
                "topic_doc_count": b_count_map.get(topic_id, np.nan),
                "keyword_rank": rank,
                "keyword": str(word),
                "keyword_score": float(score or 0.0),
            })

    bertopic_keywords_df = pd.DataFrame(b_rows)
    if not bertopic_keywords_df.empty:
        bertopic_keywords_df.to_csv(RUN_DIR / "bertopic_topic_keywords_long.csv", index=False)
        keyword_frames.append(bertopic_keywords_df)

if final_lda is not None:
    l_rows = []
    dominant_topic_ids = []
    lda_corpus_for_count = [final_lda["trainer"].dictionary.doc2bow(str(doc).split()) for doc in lda_docs]
    for bow in lda_corpus_for_count:
        if not bow:
            continue
        dist = final_lda["trainer"].model.get_document_topics(bow, minimum_probability=0.0)
        if dist:
            dominant_topic_ids.append(int(max(dist, key=lambda x: x[1])[0]))
    l_count_map = dict(Counter(dominant_topic_ids))

    for topic_id in range(final_lda["trainer"].model.num_topics):
        for rank, (word, score) in enumerate(final_lda["trainer"].model.show_topic(topic_id, topn=15), start=1):
            l_rows.append({
                "model": "LDA",
                "topic_id": int(topic_id),
                "topic_doc_count": int(l_count_map.get(int(topic_id), 0)),
                "keyword_rank": rank,
                "keyword": str(word),
                "keyword_score": float(score or 0.0),
            })

    lda_keywords_df = pd.DataFrame(l_rows)
    if not lda_keywords_df.empty:
        lda_keywords_df.to_csv(RUN_DIR / "lda_topic_keywords_long.csv", index=False)
        keyword_frames.append(lda_keywords_df)

if keyword_frames:
    combined_keywords_df = pd.concat(keyword_frames, ignore_index=True)
    combined_keywords_df.to_csv(RUN_DIR / "topic_keywords_combined_long.csv", index=False)

metrics_rows = []
if final_bertopic is not None:
    metrics_rows.append({
        "model": "BERTopic",
        "scenario_id": final_bertopic["scenario_id"],
        "coherence_cv": final_bertopic["metrics"].get("coherence_cv"),
        "topic_diversity": final_bertopic["metrics"].get("topic_diversity"),
        "num_topics": final_bertopic["metrics"].get("num_topics"),
        "model_path": final_bertopic["model_path"],
    })
if final_lda is not None:
    metrics_rows.append({
        "model": "LDA",
        "scenario_id": final_lda["scenario_id"],
        "coherence_cv": final_lda["metrics"].get("coherence_cv"),
        "topic_diversity": final_lda["metrics"].get("topic_diversity"),
        "num_topics": final_lda["metrics"].get("num_topics"),
        "model_path": final_lda["model_path"],
    })

final_metrics_df = pd.DataFrame(metrics_rows)
final_metrics_path = RUN_DIR / "best_retrain_metrics.csv"
final_metrics_df.to_csv(final_metrics_path, index=False)

best_config_payload = {
    "timestamp": RUN_TS,
    "best_bertopic": None,
    "best_lda": None,
    "comparison_top10": leaderboard_df.head(10).to_dict(orient="records") if not leaderboard_df.empty else [],
    "artifact_dir": str(RUN_DIR),
}

if final_bertopic is not None:
    best_config_payload["best_bertopic"] = {
        "scenario_id": final_bertopic["scenario_id"],
        "params": final_bertopic["params"].model_dump(),
        "metrics": final_bertopic["metrics"],
        "model_path": final_bertopic["model_path"],
    }

if final_lda is not None:
    best_config_payload["best_lda"] = {
        "scenario_id": final_lda["scenario_id"],
        "params": final_lda["params"].model_dump(),
        "metrics": final_lda["metrics"],
        "model_path": final_lda["model_path"],
    }

best_config_path = RUN_DIR / "best_config.json"
write_json(best_config_path, best_config_payload)

display(final_metrics_df)
print("Best config saved to:", best_config_path)

## 5) Visualisasi + DTA (Emerging/Stable/Declining)

In [ ]:
def plot_wordcloud_grid(topic_payloads, title_text, max_topics=8, cols=3):
    if WordCloud is None:
        print(f"{title_text} dilewati: package wordcloud belum tersedia.")
        return
    payloads = [p for p in topic_payloads[:max_topics] if p.get("freq")]
    if not payloads:
        print(f"{title_text} dilewati: tidak ada topik valid.")
        return

    n = len(payloads)
    cols = max(1, min(cols, n))
    rows = int(math.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, payload in zip(axes, payloads):
        wc = WordCloud(
            width=1400,
            height=700,
            background_color="white",
            colormap="viridis",
            max_words=200,
        ).generate_from_frequencies(payload["freq"])
        ax.imshow(wc, interpolation="bilinear")
        ax.set_title(payload.get("title", ""), fontsize=10)
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(title_text, fontsize=14)
    plt.tight_layout()
    plt.show()


print("=== Visualisasi BERTopic ===")
if final_bertopic is not None:
    b_model = final_bertopic["trainer"].model
    b_info = b_model.get_topic_info().copy()
    if "Topic" in b_info.columns:
        b_info = b_info[b_info["Topic"] != -1].copy()

    if not b_info.empty:
        cols_show = [c for c in ["Topic", "Count", "Name"] if c in b_info.columns]
        display(b_info[cols_show].head(15))

        if "Count" in b_info.columns:
            top_df = (
                b_info.sort_values("Count", ascending=False)
                .head(min(15, len(b_info)))
                .sort_values("Count")
            )
            plt.figure(figsize=(10, 6))
            plt.barh(top_df["Topic"].astype(str), top_df["Count"], color="#1f77b4")
            plt.title("BERTopic: Distribusi Dokumen per Topik")
            plt.xlabel("Jumlah Dokumen")
            plt.ylabel("Topic ID")
            plt.tight_layout()
            plt.show()

        wc_payloads = []
        if "Count" in b_info.columns:
            topic_ids = b_info.sort_values("Count", ascending=False)["Topic"].astype(int).head(8).tolist()
        else:
            topic_ids = b_info["Topic"].astype(int).head(8).tolist()

        for tid in topic_ids:
            words_scores = b_model.get_topic(int(tid)) or []
            freq = {w: float(s or 0.0) for w, s in words_scores[:15] if float(s or 0.0) > 0}
            wc_payloads.append({"title": f"Topic {int(tid)}", "freq": freq})

        plot_wordcloud_grid(wc_payloads, "BERTopic WordCloud per Topic", max_topics=8, cols=3)
    else:
        print("Topic info BERTopic kosong setelah filter outlier.")
else:
    print("Best BERTopic tidak tersedia.")

print("\n=== Visualisasi LDA ===")
if final_lda is not None:
    l_trainer = final_lda["trainer"]
    l_model = l_trainer.model

    lda_topic_rows = []
    for topic_id in range(l_model.num_topics):
        topic_words = l_model.show_topic(topic_id, topn=15)
        lda_topic_rows.append({
            "topic_id": int(topic_id),
            "top_words": ", ".join([w for w, _ in topic_words]),
        })

    lda_topic_df = pd.DataFrame(lda_topic_rows)
    display(lda_topic_df.head(15))

    lda_bows = [l_trainer.dictionary.doc2bow(str(doc).split()) for doc in lda_docs]
    dominant_topic_ids = []
    for bow in lda_bows:
        if not bow:
            continue
        dist = l_model.get_document_topics(bow, minimum_probability=0.0)
        if dist:
            dominant_topic_ids.append(int(max(dist, key=lambda x: x[1])[0]))

    topic_counts = Counter(dominant_topic_ids)
    if topic_counts:
        lda_count_df = pd.DataFrame({
            "topic_id": list(topic_counts.keys()),
            "doc_count": list(topic_counts.values()),
        }).sort_values("doc_count", ascending=False)

        plt.figure(figsize=(10, 6))
        plt.bar(
            lda_count_df["topic_id"].astype(str),
            lda_count_df["doc_count"],
            color="#ff7f0e",
        )
        plt.title("LDA: Distribusi Dokumen per Topik Dominan")
        plt.xlabel("Topic ID")
        plt.ylabel("Jumlah Dokumen")
        plt.tight_layout()
        plt.show()

        wc_topic_ids = lda_count_df["topic_id"].astype(int).head(8).tolist()
    else:
        wc_topic_ids = list(range(min(8, l_model.num_topics)))

    wc_payloads = []
    for tid in wc_topic_ids:
        words_scores = l_model.show_topic(int(tid), topn=15)
        freq = {w: float(s or 0.0) for w, s in words_scores if float(s or 0.0) > 0}
        wc_payloads.append({"title": f"Topic {int(tid)}", "freq": freq})

    plot_wordcloud_grid(wc_payloads, "LDA WordCloud per Topic", max_topics=8, cols=3)
else:
    print("Best LDA tidak tersedia.")

print("\n=== Perbandingan Metrics Best Model ===")
if not final_metrics_df.empty:
    display(final_metrics_df)
    metrics_plot_df = final_metrics_df.copy()
    for col in ["coherence_cv", "topic_diversity"]:
        metrics_plot_df[col] = pd.to_numeric(metrics_plot_df[col], errors="coerce")

    x = np.arange(len(metrics_plot_df))
    width = 0.35

    plt.figure(figsize=(9, 5))
    plt.bar(x - width / 2, metrics_plot_df["coherence_cv"], width=width, label="Coherence c_v")
    plt.bar(x + width / 2, metrics_plot_df["topic_diversity"], width=width, label="Topic Diversity")
    plt.xticks(x, metrics_plot_df["model"].tolist())
    plt.ylabel("Score")
    plt.title("Best BERTopic vs Best LDA")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada metrics best model.")


print("\n=== DTA BERTopic: Emerging / Stable / Declining ===")
dta_paths = []
if final_bertopic is None or timestamps is None:
    print("DTA dilewati: best BERTopic atau timestamps tidak tersedia.")
else:
    dta_model = final_bertopic["trainer"].model
    time_index = pd.to_datetime(pd.Series(timestamps).astype(int).astype(str) + "-01-01", errors="coerce")

    if time_index.isna().any():
        print("DTA dilewati: ada timestamp tidak valid.")
    else:
        try:
            dta_df = dta_model.topics_over_time(bertopic_docs, timestamps=time_index.tolist())
            if dta_df.empty:
                print("DTA menghasilkan dataframe kosong.")
            else:
                dta_df = dta_df.sort_values(["Topic", "Timestamp"]).reset_index(drop=True)
                dta_raw_path = RUN_DIR / "dta_topics_over_time.csv"
                dta_df.to_csv(dta_raw_path, index=False)
                dta_paths.append(dta_raw_path)

                trend_threshold = 0.03
                min_points = 3
                trend_rows = []

                work = dta_df.copy()
                work["Topic"] = pd.to_numeric(work["Topic"], errors="coerce")
                work = work[work["Topic"].notna()].copy()
                work["Topic"] = work["Topic"].astype(int)
                work = work[work["Topic"] != -1].copy()
                work["Frequency"] = pd.to_numeric(work["Frequency"], errors="coerce").fillna(0.0)

                for topic_id, g in work.groupby("Topic"):
                    g = g.sort_values("Timestamp")
                    y = g["Frequency"].to_numpy(dtype=float)
                    x = np.arange(len(g), dtype=float)
                    n_points = len(g)

                    if n_points < min_points:
                        trend_label = "insufficient_data"
                        slope = np.nan
                        relative_slope = np.nan
                    else:
                        slope = float(np.polyfit(x, y, 1)[0])
                        baseline = max(float(np.mean(y)), 1.0)
                        relative_slope = float(slope / baseline)

                        if relative_slope >= trend_threshold and float(y[-1]) >= float(y[0]):
                            trend_label = "emerging"
                        elif relative_slope <= -trend_threshold and float(y[-1]) <= float(y[0]):
                            trend_label = "declining"
                        else:
                            trend_label = "stable"

                    topic_words = dta_model.get_topic(int(topic_id)) or []
                    keywords = ", ".join([w for w, _ in topic_words[:10]])

                    trend_rows.append({
                        "topic_id": int(topic_id),
                        "trend_label": trend_label,
                        "n_periods": int(n_points),
                        "start_frequency": float(y[0]) if n_points else np.nan,
                        "end_frequency": float(y[-1]) if n_points else np.nan,
                        "slope": slope,
                        "relative_slope": relative_slope,
                        "keywords_top10": keywords,
                    })

                trend_df = pd.DataFrame(trend_rows)
                trend_df = trend_df.sort_values(["trend_label", "relative_slope"], ascending=[True, False], na_position="last").reset_index(drop=True)

                trend_path = RUN_DIR / "dta_topic_trend_classification.csv"
                trend_df.to_csv(trend_path, index=False)
                dta_paths.append(trend_path)

                print("Ringkasan trend:")
                summary_df = trend_df.groupby("trend_label", dropna=False).size().reset_index(name="n_topics")
                display(summary_df)
                display(trend_df)

                color_map = {
                    "emerging": "#2ca02c",
                    "stable": "#1f77b4",
                    "declining": "#d62728",
                    "insufficient_data": "#7f7f7f",
                }
                colors = [color_map.get(str(v), "#7f7f7f") for v in summary_df["trend_label"].astype(str)]
                plt.figure(figsize=(8, 4))
                plt.bar(summary_df["trend_label"].astype(str), summary_df["n_topics"], color=colors)
                plt.title("Jumlah Topik per Kategori Trend DTA")
                plt.xlabel("Kategori Trend")
                plt.ylabel("Jumlah Topik")
                plt.tight_layout()
                plt.show()

                top_topics = (
                    work.groupby("Topic", as_index=False)["Frequency"]
                    .sum()
                    .sort_values("Frequency", ascending=False)
                    .head(8)["Topic"]
                    .astype(int)
                    .tolist()
                )

                plt.figure(figsize=(12, 5))
                for tid in top_topics:
                    g = work[work["Topic"] == int(tid)].sort_values("Timestamp")
                    plt.plot(
                        pd.to_datetime(g["Timestamp"], errors="coerce"),
                        g["Frequency"],
                        marker="o",
                        linewidth=1.8,
                        alpha=0.85,
                        label=f"Topic {int(tid)}",
                    )
                plt.title("Fluktuasi Frekuensi Topik DTA (Top 8)")
                plt.xlabel("Waktu")
                plt.ylabel("Frequency")
                plt.xticks(rotation=30, ha="right")
                plt.grid(alpha=0.25)
                plt.legend(loc="best", ncol=2, fontsize=8)
                plt.tight_layout()
                plt.show()

        except Exception as exc:
            print("DTA gagal dijalankan:", exc)

if dta_paths:
    print("DTA artifacts:")
    for p in dta_paths:
        print("-", p)

In [ ]:
artifact_rows = []
for p in sorted(RUN_DIR.glob("*")):
    if p.is_file():
        artifact_rows.append({
            "file": p.name,
            "size_kb": round(p.stat().st_size / 1024.0, 2),
        })

artifact_df = pd.DataFrame(artifact_rows).sort_values("file").reset_index(drop=True)
print("Artifact directory:", RUN_DIR)
display(artifact_df)